# 전처리 최종 결과 검증

2017-01-31까지 관측된 데이터로 생성한 최종 모델링 테이블을 검증한다.

확인 항목:

- 데이터 크기
- 결측치와 중복
- 사용자 중복
- 데이터 분할
- 이탈률
- 관측 기준일
- 시간 관련 피처의 범위

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
DATA_PATH = PROCESSED_DIR / "model_table_final.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("shape:", df.shape)
print("전체 결측치:", df.isna().sum().sum())
print("전체 행 중복:", df.duplicated().sum())
print("msno 중복:", df["msno"].duplicated().sum())

print("\n관측 기준일:")
print(df["snapshot"].value_counts(dropna=False))

print("\nsplit:")
print(df["split"].value_counts(dropna=False))

print("\n이탈률:")
print(df.groupby("split")["is_churn"].agg(["count", "mean"]))

shape: (992931, 44)
전체 결측치: 0
전체 행 중복: 0
msno 중복: 0

관측 기준일:
snapshot
2017-01-31    992931
Name: count, dtype: int64

split:
split
train    695051
valid    148940
test     148940
Name: count, dtype: int64

이탈률:
        count      mean
split                  
test   148940  0.063918
train  695051  0.063923
valid  148940  0.063925


In [3]:
time_features = [
    "days_since_last_txn",
    "days_to_expire",
]

print(df[time_features].describe().T)

print(
    "\ndays_since_last_txn 음수:",
    (df["days_since_last_txn"] < 0).sum()
)

print(
    "관측일 기준 이미 만료된 사용자:",
    (df["days_to_expire"] < 0).sum()
)

print(
    "관측일 기준 2월 중 만료 예정:",
    df["days_to_expire"].between(1, 28).sum()
)

print(
    "2월 이후 만료 예정:",
    (df["days_to_expire"] > 28).sum()
)

                        count       mean        std     min  25%   50%   75%  \
days_since_last_txn  992931.0  19.914687  43.196844     0.0  5.0  14.0  23.0   
days_to_expire       992931.0  17.106736  19.720089 -1029.0  9.0  17.0  26.0   

                       max  
days_since_last_txn  760.0  
days_to_expire        59.0  

days_since_last_txn 음수: 0
관측일 기준 이미 만료된 사용자: 7771
관측일 기준 2월 중 만료 예정: 887749
2월 이후 만료 예정: 90816


In [4]:
import numpy as np

df["expiry_status"] = np.select(
    [
        df["days_to_expire"] < 1,
        df["days_to_expire"].between(1, 28),
        df["days_to_expire"] > 28,
    ],
    [
        "관측일 이전·당일 만료",
        "2월 만료",
        "3월 이후 만료",
    ],
    default="기타",
)

expiry_audit = (
    df.groupby("expiry_status", observed=True)
      .agg(
          customers=("msno", "count"),
          churn_rate=("is_churn", "mean"),
          min_days=("days_to_expire", "min"),
          max_days=("days_to_expire", "max"),
      )
)

expiry_audit["customer_pct"] = (
    expiry_audit["customers"] / len(df) * 100
)

expiry_audit.sort_values("customers", ascending=False)

,customers,churn_rate,min_days,max_days,customer_pct
expiry_status,,,,,
2월 만료,887749,0.058333,1.0,28.0,89.406917
3월 이후 만료,90816,0.047128,29.0,59.0,9.146255
관측일 이전·당일 만료,14366,0.515523,-1029.0,0.0,1.446828


In [5]:
from sklearn.metrics import roc_auc_score

features = [
    "days_to_expire",
    "days_since_last_txn",
    "auto_renew_rate",
    "last_is_auto_renew",
    "txn_count",
    "d7_active_days",
]

rows = []

for feature in features:
    auc = roc_auc_score(df["is_churn"], df[feature])
    
    rows.append({
        "feature": feature,
        "auc": auc,
        "directional_auc": max(auc, 1 - auc),
        "direction": (
            "값이 클수록 이탈"
            if auc >= 0.5
            else "값이 작을수록 이탈"
        ),
    })

pd.DataFrame(rows).sort_values(
    "directional_auc",
    ascending=False
)

,feature,auc,directional_auc,direction
2,auto_renew_rate,0.251449,0.748551,값이 작을수록 이탈
3,last_is_auto_renew,0.254414,0.745586,값이 작을수록 이탈
4,txn_count,0.322478,0.677522,값이 작을수록 이탈
1,days_since_last_txn,0.629655,0.629655,값이 클수록 이탈
0,days_to_expire,0.409630,0.590370,값이 작을수록 이탈
5,d7_active_days,0.482148,0.517852,값이 작을수록 이탈


In [6]:
from pandas.api.types import is_numeric_dtype
from sklearn.metrics import roc_auc_score

exclude_cols = {
    "is_churn",
}

rows = []

for feature in df.columns:
    if feature in exclude_cols:
        continue

    if not is_numeric_dtype(df[feature]):
        continue

    if df[feature].nunique(dropna=False) <= 1:
        continue

    auc = roc_auc_score(
        df["is_churn"],
        df[feature]
    )

    rows.append({
        "feature": feature,
        "auc": auc,
        "directional_auc": max(auc, 1 - auc),
        "direction": (
            "값이 클수록 이탈"
            if auc >= 0.5
            else "값이 작을수록 이탈"
        ),
    })

univariate_auc = (
    pd.DataFrame(rows)
      .sort_values("directional_auc", ascending=False)
)

univariate_auc.head(20)

,feature,auc,directional_auc,direction
8,auto_renew_rate,0.251449,0.748551,값이 작을수록 이탈
16,last_is_auto_renew,0.254414,0.745586,값이 작을수록 이탈
14,last_plan_list_price,0.679462,0.679462,값이 클수록 이탈
5,txn_count,0.322478,0.677522,값이 작을수록 이탈
15,last_actual_amount_paid,0.671778,0.671778,값이 클수록 이탈
17,days_since_last_txn,0.629655,0.629655,값이 클수록 이탈
11,total_amount_paid,0.407696,0.592304,값이 작을수록 이탈
18,days_to_expire,0.409630,0.590370,값이 작을수록 이탈
3,bd_is_missing,0.419724,0.580276,값이 작을수록 이탈
6,payment_method_nunique,0.576148,0.576148,값이 클수록 이탈
